# 放射組學遺傳演算法（GA）特徵選擇與分類實驗室 (Kaggle 優化自適應版)

這個 Notebook 是一個**完全獨立且自包含（Self-contained）**的遺傳演算法特徵選擇與分類流水線。它專為 Kaggle 線上環境設計，**自動適應 Kaggle 的輸入與工作輸出路徑**，無需修改任何外部 Python 原始碼，便能完成全自動的數據特徵篩選、模型訓練評估與視覺化進程記錄。

## 🌟 核心功能與升級特點：
1. **Kaggle 零設定自適應**：自動偵測 Kaggle 資料集路徑 `/kaggle/input/datasets/sungjinyi/resampled-feature`，並將所有產出自動輸出到 `/kaggle/working/`。
2. **10 個 BinWidths 迴圈遍歷**：自動搜尋、排序並依次處理目錄下的 10 個組寬特徵檔案。
3. **強大中斷點續傳 (Resumability)**：自動偵測每個 binWidth 目錄下的 `results_log.csv` 是否已存在。若是，則自動跳過，支援在離線或斷線後再次重啟執行。
4. **多分類器支援 (StandardScaler 整合)**：支援隨機森林、SVM、以及 XGBoost 的動態選擇。在每一折訓練集內部自適應執行 `StandardScaler` 正規化，完全防止 Leakage，為 SVM 等敏感模型提供穩健的數值支持。
5. **多 GA 方法循環與 DeLong 基線對比**：支援一次運行多個 GA 篩選方法，並在結束後，對每個 GA 方法與傳統 MI 基線進行學術級的 DeLong ROC 檢定，輸出 `delong_results.csv`。
6. **世代 AUC 軌跡保存與繪圖**：記錄每個進化世代最優特徵組合的 AUC 得分，並將繪製的學術進化曲線自動儲存為 PNG 圖片。

In [ ]:
# ─── Kaggle 依賴安裝 ───
# 如果您在 Kaggle 上執行，請執行此儲存格以安裝 leidenalg 等必要的特殊圖論聚類套件
!pip install python-igraph leidenalg xgboost matplotlib pandas numpy

  Using cached texttable-1.7.0-py2.py3-none-any.whl.metadata (9.8 kB)


   ━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.3/2.7 MB ? eta -:--:--

   ━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.5/2.7 MB 1.9 MB/s eta 0:00:02

   ━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━ 1.6/2.7 MB 1.8 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━ 2.1/2.7 MB 1.8 MB/s eta 0:00:01

...

   ━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━ 45.4/98.7 MB 2.2 MB/s eta 0:00:25

   ━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━ 46.1/98.7 MB 2.2 MB/s eta 0:00:25

   ━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━ 47.2/98.7 MB 2.2 MB/s eta 0:00:24

   ━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━ 48.2/98.7 MB 2.2 MB/s eta 0:00:24

   ━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━ 49.3/98.7 MB 2.2 MB/s eta 0:00:23

In [ ]:
# ─── 核心導入 ───
import os
import re
import sys
import glob
import time
import json
import pickle
import warnings
import subprocess
from pathlib import Path
from joblib import Parallel, delayed

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

from sklearn.feature_selection import VarianceThreshold, mutual_info_classif, f_classif
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

import igraph as ig
import leidenalg

warnings.filterwarnings('ignore')
print("所有必要庫導入與驗證成功！")

## ⚙️ 全域參數與自適應路徑配置區

在此處，系統會自動偵測是否身處 Kaggle 環境，並動態綁定最優輸入與輸出位置。您可以自由修改 `CLASSIFIER_TYPE` 與要執行的 `METHODS_TO_RUN`。

In [ ]:
# ─── 全域自適應路徑配置 ───
KAGGLE_INPUT_DIR = "/kaggle/input/datasets/sungjinyi/resampled-70-cases-bw50"
KAGGLE_OUTPUT_DIR = "/kaggle/working/result"

if os.path.exists(KAGGLE_INPUT_DIR):
    INPUT_DIR = KAGGLE_INPUT_DIR
    OUTPUT_DIR = KAGGLE_OUTPUT_DIR
else:
    # 本地或備用環境 fallback
    INPUT_DIR = "test_resampled/feature_resampled"
    OUTPUT_DIR = "test_resampled/result"

# ─── 實驗超參數配置 ───
# 1. 選擇分類器種類。可選值：'rf' (隨機森林), 'svm' (SVC 支持向量機), 'xgboost' (XGBoost)
CLASSIFIER_TYPE = 'svm'

# 2. 特徵限制與懲罰係數
MAX_K = 20
CORR_THRESHOLD = 0.7
LAMBDA_PENALTY = 0.005

# 3. GA 超參數（全量模式與快速測試模式會自動重寫此處）
POP_SIZE = 30
N_GENERATIONS = 30
PATIENCE = 10
CROSSOVER_RATE = 0.8
MUTATION_RATE = 0.05

# ─── CPU 平行化配置 ───
N_CPU_WORKERS = min(4, os.cpu_count() or 1)
print(f"[CPU 平行化] 將使用 {N_CPU_WORKERS} 個工作程序 (Worker) 進行平行運算。")

# ─── 預定義內置 GA 方法配置 ───
# 最後一個Cell修改 METHODS_TO_RUN
METHOD_CONFIGS = {
    'MI_top_K': {'fitness': 'MI', 'constraint': 'none', 'soft_penalty': True},
    'MI_unrestricted': {'fitness': 'MI', 'constraint': 'none', 'soft_penalty': True},
    'AUC_unrestricted': {'fitness': 'AUC', 'constraint': 'none', 'soft_penalty': True},
    'MI_local_search': {'fitness': 'MI', 'constraint': 'local_search', 'soft_penalty': False},
    'AUC_local_search': {'fitness': 'AUC', 'constraint': 'local_search', 'soft_penalty': False},
    'MI_community': {'fitness': 'MI', 'constraint': 'community', 'soft_penalty': False, 'theta': 0.7, 'omega': 3},
    'AUC_community': {'fitness': 'AUC', 'constraint': 'community', 'soft_penalty': False, 'theta': 0.7, 'omega': 3},
    'MI_enforce': {'fitness': 'MI', 'constraint': 'enforce', 'soft_penalty': False},
    'AUC_enforce': {'fitness': 'AUC', 'constraint': 'enforce', 'soft_penalty': False},
}

print(f"[環境偵測成功] 輸入目錄: {INPUT_DIR} | 輸出目錄: {OUTPUT_DIR}")
print(f"[當前分類器] {CLASSIFIER_TYPE.upper()}")

## 📦 數據載入、前置過濾與 Z-score 標準化

為確保嚴格的零洩漏防線，我們將 `StandardScaler` 的 `fit_transform` 限制在 `prefilter` 之後的訓練集 `X_tr` 內部，測試集 `X_te` 僅調用 `transform`。

In [ ]:
def load_data(file_path):
    """載入特定組寬的放射組學數據。"""
    df = pd.read_csv(file_path)
    case_col = None
    for col in ["CaseNumber", "CaseID"]:
        if col in df.columns:
            case_col = col
            break
    if case_col is None:
        raise ValueError("數據中找不到 CaseNumber 欄位")
    X = df.drop(columns=[case_col, 'Label']).values.astype(np.float64)
    y = df['Label'].values.astype(np.int32)
    return X, y, df[case_col].values

def prefilter(X_train, X_test, y_train, corr_threshold=CORR_THRESHOLD):
    """訓練折內的前置特徵過濾與特徵選擇。"""
    vt = VarianceThreshold(threshold=0)
    vt.fit(X_train)
    vt_mask = vt.get_support()
    X_tr = X_train[:, vt_mask]
    X_te = X_test[:, vt_mask]
    
    corr_matrix = np.abs(np.corrcoef(X_tr.T))
    n_vt = X_tr.shape[1]
    to_drop = set()
    for a in range(n_vt):
        if a in to_drop:
            continue
        for b in range(a + 1, n_vt):
            if b in to_drop:
                continue
            if corr_matrix[a, b] > corr_threshold:
                to_drop.add(b)
                
    keep_mask = np.array([i not in to_drop for i in range(n_vt)])
    X_tr = X_tr[:, keep_mask]
    X_te = X_te[:, keep_mask]
    
    mi_scores = mutual_info_classif(X_tr, y_train, random_state=42)
    return X_tr, X_te, mi_scores, vt_mask, keep_mask

## 🤖 多分類器動態工廠與評估模組

In [ ]:
def get_classifier(classifier_type, random_state=42, inner=True):
    """配合多分類器動態切換支援。"""
    if classifier_type == 'rf':
        return RandomForestClassifier(
            n_estimators=100 if inner else 500,
            max_depth=5 if inner else None,
            random_state=random_state,
            n_jobs=1
        )
    elif classifier_type == 'svm':
        return SVC(
            C=1.0, kernel='rbf', probability=True, random_state=random_state
        )
    elif classifier_type == 'xgboost':
        return XGBClassifier(
            n_estimators=100 if inner else 200,
            max_depth=5 if inner else 6,
            random_state=random_state,
            use_label_encoder=False,
            eval_metric='logloss',
            n_jobs=1
        )
    else:
        raise ValueError(f"不支援的分類器類型: {classifier_type}")

def evaluate_fold(X_tr_final, X_te_final, y_train, y_test, classifier_type):
    """利用選定分類器的外層最優配置對測試集進行最終評估。"""
    clf = get_classifier(classifier_type, random_state=42, inner=False)
    clf.fit(X_tr_final, y_train)
    y_score = clf.predict_proba(X_te_final)[:, 1]
    
    try:
        auc_val = roc_auc_score(y_test, y_score)
    except ValueError:
        auc_val = np.nan
        
    y_pred = (y_score >= 0.5).astype(int)
    acc_val = accuracy_score(y_test, y_pred)
    
    tp = np.sum((y_pred == 1) & (y_test == 1))
    tn = np.sum((y_pred == 0) & (y_test == 0))
    fp = np.sum((y_pred == 1) & (y_test == 0))
    fn = np.sum((y_pred == 0) & (y_test == 1))
    
    sensitivity = tp / max(tp + fn, 1)
    specificity = tn / max(tn + fp, 1)
    
    return {
        'auc': auc_val,
        'acc': acc_val,
        'sensitivity': sensitivity,
        'specificity': specificity,
        'y_true': y_test,
        'y_score': y_score,
    }

## 🎯 適應度函數與世代 AUC 追蹤設計

In [ ]:
def spawn_rng_seeds(entropy, n_workers):
    """使用 SeedSequence 為每個 worker 生成獨立的 RNG 子種子。"""
    ss = np.random.SeedSequence(entropy)
    child_seeds = ss.spawn(n_workers)
    return child_seeds

In [ ]:
def mi_fitness(chromosome, mi_scores, X_tr, soft_penalty=False, lambda_p=LAMBDA_PENALTY):
    selected = np.where(chromosome == 1)[0]
    n_sel = len(selected)
    if n_sel < 2:
        return 0.0
    relevance = np.mean(mi_scores[selected])
    corr_sel = np.abs(np.corrcoef(X_tr[:, selected].T))
    np.fill_diagonal(corr_sel, 0)
    mean_pair_corr = corr_sel.sum() / (n_sel * (n_sel - 1))
    distinct = 1.0 / (1.0 + mean_pair_corr)
    fitness = relevance + 0.3 * distinct
    if soft_penalty:
        fitness -= lambda_p * max(0, n_sel - MAX_K)
    return fitness

def auc_fitness(chromosome, X_tr, y_tr, inner_cv, classifier_type, soft_penalty=False, lambda_p=LAMBDA_PENALTY):
    selected = np.where(chromosome == 1)[0]
    n_sel = len(selected)
    if n_sel < 2:
        return 0.0
    try:
        scores = []
        for tr_idx, va_idx in inner_cv.split(X_tr[:, selected], y_tr):
            clf = get_classifier(classifier_type, random_state=42, inner=True)
            clf.fit(X_tr[np.ix_(tr_idx, selected)], y_tr[tr_idx])
            try:
                s = roc_auc_score(y_tr[va_idx], clf.predict_proba(X_tr[np.ix_(va_idx, selected)])[:, 1])
            except ValueError:
                s = 0.5
            scores.append(s)
        scaled_auc = max(0.0, (np.mean(scores) - 0.5) * 2.0)
    except Exception:
        return 0.0
    corr_sel = np.abs(np.corrcoef(X_tr[:, selected].T))
    np.fill_diagonal(corr_sel, 0)
    avg_red = corr_sel.sum() / (n_sel * (n_sel - 1)) if n_sel > 1 else 0.0
    fitness = (scaled_auc ** 2) / (1.0 + avg_red)
    if soft_penalty:
        fitness -= lambda_p * max(0, n_sel - MAX_K)
    return fitness

def get_chromosome_auc(chromosome, X_tr, y_tr, inner_cv, classifier_type):
    selected = np.where(chromosome == 1)[0]
    if len(selected) < 2:
        return 0.5
    try:
        scores = []
        for tr_idx, va_idx in inner_cv.split(X_tr[:, selected], y_tr):
            clf = get_classifier(classifier_type, random_state=42, inner=True)
            clf.fit(X_tr[np.ix_(tr_idx, selected)], y_tr[tr_idx])
            try:
                s = roc_auc_score(y_tr[va_idx], clf.predict_proba(X_tr[np.ix_(va_idx, selected)])[:, 1])
            except ValueError:
                s = 0.5
            scores.append(s)
        return np.mean(scores)
    except Exception:
        return 0.5

## 🧬 遺傳算法核心算子與約束控制工廠

In [ ]:
def init_population(rng, n_features, mi_scores, pop_size=POP_SIZE, k_range=(5, 40), n_seeded=5, constraint_fn=None):
    population = np.zeros((pop_size, n_features), dtype=np.int8)
    top_mi = np.argsort(mi_scores)[-min(30, n_features):]
    for i in range(pop_size):
        k = rng.integers(k_range[0], min(k_range[1], n_features + 1))
        if i < n_seeded:
            chosen = rng.choice(top_mi, size=min(k, len(top_mi)), replace=False)
        else:
            chosen = rng.choice(n_features, size=k, replace=False)
        population[i, chosen] = 1
        if constraint_fn is not None:
            population[i] = constraint_fn(population[i])
    return population

def tournament_select(rng, fitnesses, pop_size, tournament_size=3):
    candidates = rng.choice(pop_size, size=tournament_size, replace=False)
    return candidates[np.argmax(fitnesses[candidates])]

def rank_select(rng, fitnesses, pop_size):
    ranked = np.argsort(fitnesses)[::-1]
    ranks = np.empty(pop_size, dtype=float)
    ranks[ranked] = np.arange(1, pop_size + 1, dtype=float)
    probs = ranks / ranks.sum()
    return rng.choice(pop_size, p=probs)

def crossover(rng, p1, p2, n_features, rate=CROSSOVER_RATE):
    if rng.random() < rate and n_features >= 2:
        pts = sorted(rng.choice(n_features, size=2, replace=False))
        c1, c2 = p1.copy(), p2.copy()
        c1[pts[0]:pts[1]] = p2[pts[0]:pts[1]]
        c2[pts[0]:pts[1]] = p1[pts[0]:pts[1]]
    else:
        c1, c2 = p1.copy(), p2.copy()
    return c1, c2

def mutate(rng, chromosome, n_features, rate=MUTATION_RATE):
    mask = rng.random(n_features) < rate
    chromosome[mask] = 1 - chromosome[mask]
    return chromosome

In [ ]:
def make_local_search_constraint(X_tr, mi_scores, max_k=MAX_K, m_distinct=0.65):
    corr = np.abs(np.corrcoef(X_tr.T))
    np.fill_diagonal(corr, 0)
    mean_corr = corr.mean(axis=1)
    feature_order = np.argsort(mean_corr)
    n_distinct = X_tr.shape[1] // 2
    d = max(1, int(m_distinct * max_k))
    x = max_k - d
    def constraint_fn(chrom):
        D_set = set(feature_order[:n_distinct].tolist())
        S_set = set(feature_order[n_distinct:].tolist())
        sel = set(np.where(chrom == 1)[0])
        Xd = sel & D_set
        Xs = sel & S_set
        if len(Xd) < d:
            cands = [f for f in feature_order[:n_distinct] if f not in sel and chrom[f] == 0]
            for f in cands[:d - len(Xd)]:
                chrom[f] = 1
                Xd.add(f)
        elif len(Xd) > d:
            lst = sorted(Xd, key=lambda f: feature_order.tolist().index(f) if f in feature_order.tolist() else 0, reverse=True)
            for f in lst[:len(Xd) - d]:
                chrom[f] = 0
                Xd.discard(f)
        if len(Xs) < x:
            cands = [f for f in feature_order[n_distinct:] if f not in sel and chrom[f] == 0]
            for f in cands[:x - len(Xs)]:
                chrom[f] = 1
                Xs.add(f)
        elif len(Xs) > x:
            lst = sorted(Xs, key=lambda f: feature_order.tolist().index(f) if f in feature_order.tolist() else 0, reverse=True)
            for f in lst[:len(Xs) - x]:
                chrom[f] = 0
                Xs.discard(f)
        selected = np.where(chrom == 1)[0]
        if len(selected) > max_k:
            excess = len(selected) - max_k
            mc = mean_corr[selected]
            to_remove = selected[np.argsort(mc)[-excess:]]
            chrom[to_remove] = 0
        return chrom
    return constraint_fn

def make_community_constraint(X_tr, y_tr, rng, theta=0.7, omega=3, max_k=MAX_K):
    f_scores, _ = f_classif(X_tr, y_tr)
    f_scores = np.nan_to_num(f_scores, nan=0.0, posinf=0.0, neginf=0.0)
    mean_val, std_val = np.mean(f_scores), np.std(f_scores)
    fisher_norm = (
        1.0 / (1.0 + np.exp(-(f_scores - mean_val) / std_val))
        if std_val > 0
        else np.ones_like(f_scores) * 0.5
    )
    corr_matrix = np.corrcoef(X_tr.T)
    abs_corr = np.abs(corr_matrix)
    np.fill_diagonal(abs_corr, 0.0)
    rows, cols = np.where(np.triu(abs_corr >= theta, k=1))
    edge_list = list(zip(rows.tolist(), cols.tolist()))
    weight_list = abs_corr[rows, cols].tolist()
    g = ig.Graph(n=X_tr.shape[1], edges=edge_list, directed=False)
    if len(edge_list) > 0:
        g.es["weight"] = weight_list
    if len(edge_list) == 0:
        communities = [[i] for i in range(X_tr.shape[1])]
    else:
        part = leidenalg.find_partition(g, leidenalg.ModularityVertexPartition, weights="weight")
        communities = [list(m) for m in part]
    def scoring_repair(chrom):
        for comm in communities:
            max_sel = min(omega, len(comm))
            sel_in_comm = [f for f in comm if chrom[f] == 1]
            if len(sel_in_comm) > max_sel:
                n_rem = len(sel_in_comm) - max_sel
                sc = np.maximum(fisher_norm[sel_in_comm], 1e-12)
                inv = 1.0 / sc
                probs = inv / inv.sum()
                to_rem = rng.choice(sel_in_comm, size=n_rem, replace=False, p=probs)
                chrom[to_rem] = 0
            elif len(sel_in_comm) < max_sel:
                n_add = max_sel - len(sel_in_comm)
                candidates = [f for f in comm if chrom[f] == 0]
                if len(candidates) == 0:
                    continue
                n_add = min(n_add, len(candidates))
                to_add = rng.choice(candidates, size=n_add, replace=False)
                chrom[to_add] = 1
        return chrom
    def constraint_fn(chrom):
        chrom = scoring_repair(chrom)
        selected = np.where(chrom == 1)[0]
        if len(selected) > max_k:
            excess = len(selected) - max_k
            fs = fisher_norm[selected]
            to_remove = selected[np.argsort(fs)[:excess]]
            chrom[to_remove] = 0
        return chrom
    return constraint_fn, communities, fisher_norm

def init_population_community(rng, n_features, mi_scores, communities, fisher_norm, pop_size=30, omega=3, max_k=MAX_K):
    population = np.zeros((pop_size, n_features), dtype=np.int8)
    n_community = int(pop_size * 0.8)
    for i in range(n_community):
        for comm in communities:
            k = min(rng.integers(1, omega + 1), len(comm))
            chosen = rng.choice(comm, size=k, replace=False)
            population[i, chosen] = 1
    n_seeded = pop_size - n_community
    top_fisher = np.argsort(fisher_norm)[-min(30, n_features):]
    for i in range(n_seeded):
        k = rng.integers(5, min(max_k + 1, len(top_fisher) + 1))
        chosen = rng.choice(top_fisher, size=k, replace=False)
        population[n_community + i, chosen] = 1
        for comm in communities:
            if not any(f in chosen for f in comm):
                if rng.random() < 0.5:
                    k_comm = min(rng.integers(1, omega + 1), len(comm))
                    extra = rng.choice(comm, size=k_comm, replace=False)
                    population[n_community + i, extra] = 1
    return population

def make_enforce_constraint(rng, max_k=MAX_K):
    def constraint_fn(chrom):
        selected = np.where(chrom == 1)[0]
        n_sel = len(selected)
        if n_sel > max_k:
            n_flip = n_sel - max_k
            to_flip = rng.choice(selected, size=n_flip, replace=False)
            chrom[to_flip] = 0
        return chrom
    return constraint_fn

## 🌀 進化算法主引擎 (與世代 AUC 追蹤)

In [ ]:
def evolve(rng, population, fitness_fn, n_features, X_tr, y_tr, inner_cv, classifier_type,
           pop_size=POP_SIZE, n_generations=N_GENERATIONS, patience=PATIENCE,
           select_fn='tournament', constraint_fn=None):
    fitnesses = np.array([fitness_fn(population[i]) for i in range(pop_size)])
    best_idx = np.argmax(fitnesses)
    best_chromosome = population[best_idx].copy()
    best_fitness = fitnesses[best_idx]
    stale = 0
    history = []
    auc_history = []
    _select = tournament_select if select_fn == 'tournament' else rank_select
    
    for gen in range(n_generations):
        prev_best = best_fitness
        for _ in range(pop_size // 2):
            p1_idx = _select(rng, fitnesses, pop_size)
            p2_idx = _select(rng, fitnesses, pop_size)
            c1, c2 = crossover(rng, population[p1_idx], population[p2_idx], n_features)
            c1 = mutate(rng, c1, n_features)
            c2 = mutate(rng, c2, n_features)
            if constraint_fn is not None:
                c1 = constraint_fn(c1)
                c2 = constraint_fn(c2)
            for child in [c1, c2]:
                f = fitness_fn(child)
                worst = np.argmin(fitnesses)
                if f > fitnesses[worst]:
                    population[worst] = child
                    fitnesses[worst] = f
                    if f > best_fitness:
                        best_fitness = f
                        best_chromosome = child.copy()
        n_sel = int(np.sum(best_chromosome))
        history.append((gen, best_fitness, n_sel))
        gen_auc = get_chromosome_auc(best_chromosome, X_tr, y_tr, inner_cv, classifier_type)
        auc_history.append(gen_auc)
        if best_fitness - prev_best < 1e-6:
            stale += 1
        else:
            stale = 0
        if stale >= patience:
            while len(auc_history) < n_generations:
                auc_history.append(gen_auc)
            break
    return best_chromosome, best_fitness, history, auc_history

## 📈 學術視覺化與學術指標（Youden's J & DeLong Test）

In [ ]:
def delong_roc_test(y_true, y_pred1, y_pred2):
    y_true = np.asarray(y_true)
    y_pred1 = np.asarray(y_pred1)
    y_pred2 = np.asarray(y_pred2)
    pos1 = y_pred1[y_true == 1]
    neg1 = y_pred1[y_true == 0]
    pos2 = y_pred2[y_true == 1]
    neg2 = y_pred2[y_true == 0]
    m, n = len(pos1), len(neg1)
    if m == 0 or n == 0:
        return 0.0, 1.0, 0.5, 0.5
    V10_1 = np.zeros((m, n))
    V10_2 = np.zeros((m, n))
    for i in range(m):
        V10_1[i] = (pos1[i] > neg1).astype(float) + 0.5 * (pos1[i] == neg1).astype(float)
        V10_2[i] = (pos2[i] > neg2).astype(float) + 0.5 * (pos2[i] == neg2).astype(float)
    S10_1 = V10_1.mean(axis=1)
    S10_2 = V10_2.mean(axis=1)
    S01_1 = V10_1.T.mean(axis=1)
    S01_2 = V10_2.T.mean(axis=1)
    auc1, auc2 = V10_1.mean(), V10_2.mean()
    cov10 = np.cov(S10_1, S10_2) if m > 1 else np.zeros((2, 2))
    cov01 = np.cov(S01_1, S01_2) if n > 1 else np.zeros((2, 2))
    var_auc1 = cov10[0, 0] / m + cov01[0, 0] / n
    var_auc2 = cov10[1, 1] / m + cov01[1, 1] / n
    cov_auc = cov10[0, 1] / m + cov01[0, 1] / n
    var_diff = var_auc1 + var_auc2 - 2 * cov_auc
    if var_diff <= 0:
        var_diff = 1e-10
    z = (auc1 - auc2) / np.sqrt(var_diff)
    p = 2 * stats.norm.sf(np.abs(z))
    return z, p, auc1, auc2

def plot_auc_generations(gen_aucs_per_fold, method_name, classifier_name, out_dir=None):
    plt.figure(figsize=(10, 6))
    n_gens = gen_aucs_per_fold.shape[1]
    gens = np.arange(1, n_gens + 1)
    for fold_idx in range(5):
        plt.plot(gens, gen_aucs_per_fold[fold_idx], alpha=0.55, linestyle=':',
                 label=f'Fold {fold_idx} (Final Inner CV AUC: {gen_aucs_per_fold[fold_idx][-1]:.3f})')
    mean_trajectory = np.mean(gen_aucs_per_fold, axis=0)
    plt.plot(gens, mean_trajectory, color='black', linewidth=3.0, linestyle='--',
             label=f'Average Progress (Final Mean Inner CV AUC: {mean_trajectory[-1]:.3f})')
    plt.xlabel('Generation', fontsize=12, fontweight='bold')
    plt.ylabel('Inner CV AUC', fontsize=12, fontweight='bold')
    plt.title(f'GA Evolution Progress: AUC vs Generation\nMethod: {method_name} | Classifier: {classifier_name.upper()}', 
              fontsize=14, fontweight='bold', pad=15)
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.xlim(1, n_gens)
    plt.ylim(0.45, 1.02)
    plt.legend(fontsize=10, loc='lower right')
    plt.tight_layout()
    if out_dir:
        fig_path = os.path.join(out_dir, f'evolution_auc_{method_name}.png')
        plt.savefig(fig_path, dpi=150)
        print(f"  [視覺化存檔] 進化曲線圖片已儲存至: {fig_path}")
    plt.show()

## 📝 結果保存輔助工具

In [ ]:
def save_results_csv(results_list, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    csv_path = os.path.join(output_dir, 'results_log.csv')
    rows = []
    for r in results_list:
        for fold_idx in range(len(r['fold_aucs'])):
            rows.append({
                'method': r['method'],
                'random_state': r['random_state'],
                'fold': fold_idx,
                'auc': r['fold_aucs'][fold_idx],
                'acc': r['fold_accs'][fold_idx],
                'n_selected': r['fold_n_sel'][fold_idx],
                'feature_indices': ';'.join(map(str, r['fold_features'][fold_idx])),
            })
    df_new = pd.DataFrame(rows)
    if os.path.exists(csv_path):
        df_existing = pd.read_csv(csv_path)
        df_combined = pd.concat([df_existing, df_new], ignore_index=True)
        df_combined = df_combined.drop_duplicates(subset=['method', 'random_state', 'fold'], keep='last')
        df_combined.to_csv(csv_path, index=False)
    else:
        df_new.to_csv(csv_path, index=False)
    return csv_path

def save_results_detail(results_list, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    json_path = os.path.join(output_dir, 'results_detail.json')
    serializable = []
    for r in results_list:
        entry = {k: v for k, v in r.items() if k not in ('y_true', 'y_score', 'fold_probabilities')}
        serializable.append(entry)
    if os.path.exists(json_path):
        with open(json_path, 'r') as f:
            existing = json.load(f)
        existing.extend(serializable)
        seen = set()
        deduped = []
        for e in reversed(existing):
            key = (e['method'], e['random_state'])
            if key not in seen:
                seen.add(key)
                deduped.append(e)
        deduped = list(reversed(deduped))
    else:
        deduped = serializable
    with open(json_path, 'w') as f:
        json.dump(deduped, f, indent=2)
    return json_path

## 🧪 單次分折實驗與演化運行核心

In [ ]:
def _worker_single_rs(method, X, y, random_state, child_seed, classifier_type, out_dir=None):
    """處理單一 (方法, 隨機種子) 組合的完整 5-Fold 交叉驗證與 GA 演化。"""
    rng = np.random.default_rng(child_seed)  # 使用 SeedSequence 子種子
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)
    fold_aucs, fold_accs, fold_n_sel = [], [], []
    fold_features = []
    all_y_true, all_y_score = [], []
    gen_aucs_per_fold = []
    fold_probabilities = []

    for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        # 訓練集內前置特徵過濾，杜絕 Leakage
        X_tr, X_te, mi_scores, vt_mask, keep_mask = prefilter(X_train, X_test, y_train)
        
        # 特徵標準化 (Standardization / Scaling) 確保 SVM 等特徵數值區間統一，杜絕 Leakage
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_tr)
        X_te = scaler.transform(X_te)
        
        n_features = X_tr.shape[1]
        
        config = METHOD_CONFIGS.get(method, {'fitness': 'MI', 'constraint': 'none', 'soft_penalty': True})
        fitness_type = config['fitness']
        constraint_type = config['constraint']
        soft_penalty = config.get('soft_penalty', True)
        
        inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=random_state)
        
        constraint_fn = None
        communities, fisher_norm = None, None
        community_omega = None
        if constraint_type == 'local_search':
            constraint_fn = make_local_search_constraint(X_tr, mi_scores, max_k=MAX_K, m_distinct=0.65)
        elif constraint_type == 'community':
            theta = config.get('theta', 0.7)
            omega = config.get('omega', 3)
            community_omega = omega
            constraint_fn, communities, fisher_norm = make_community_constraint(
                X_tr, y_train, rng, theta=theta, omega=omega, max_k=MAX_K
            )
        elif constraint_type == 'enforce':
            constraint_fn = make_enforce_constraint(rng, max_k=MAX_K)

        if fitness_type == 'MI':
            def fitness_fn(chrom):
                return mi_fitness(chrom, mi_scores, X_tr, soft_penalty=soft_penalty)
        else:
            def fitness_fn(chrom):
                return auc_fitness(chrom, X_tr, y_train, inner_cv, classifier_type, soft_penalty=soft_penalty)

        k_lo = max(5, MAX_K - 10)
        k_hi = min(MAX_K + 15, n_features + 1) if constraint_type == 'none' else min(MAX_K + 1, n_features + 1)
        
        if constraint_type == 'community':
            population = init_population_community(
                rng, n_features, mi_scores, communities, fisher_norm,
                pop_size=POP_SIZE, omega=community_omega, max_k=MAX_K
            )
        else:
            population = init_population(
                rng, n_features, mi_scores, pop_size=POP_SIZE,
                k_range=(k_lo, k_hi), constraint_fn=constraint_fn
            )

        # 🧬 啟動進化優化與世代 AUC 追蹤
        best_chromosome, best_fitness, history, fold_auc_history = evolve(
            rng, population, fitness_fn, n_features, X_tr, y_train, inner_cv, classifier_type,
            pop_size=POP_SIZE, n_generations=N_GENERATIONS, patience=PATIENCE,
            select_fn='tournament', constraint_fn=constraint_fn
        )
        
        gen_aucs_per_fold.append(fold_auc_history)
        
        selected = np.where(best_chromosome == 1)[0]
        X_tr_final = X_tr[:, selected]
        X_te_final = X_te[:, selected]
        
        result = evaluate_fold(X_tr_final, X_te_final, y_train, y_test, classifier_type)
        fold_aucs.append(result['auc'])
        fold_accs.append(result['acc'])
        fold_n_sel.append(len(selected))
        fold_features.append(selected.tolist())
        
        all_y_true.extend(y_test.tolist())
        all_y_score.extend(result['y_score'].tolist())
        
        fold_probabilities.append({
            'fold': fold_idx,
            'test_indices': test_idx.tolist(),
            'y_true': y_test.tolist(),
            'y_score': result['y_score'].tolist(),
        })
        
    return {
        'method': method,
        'random_state': random_state,
        'fold_aucs': fold_aucs,
        'fold_accs': fold_accs,
        'fold_n_sel': fold_n_sel,
        'fold_features': fold_features,
        'mean_auc': np.nanmean(fold_aucs),
        'mean_acc': np.mean(fold_accs),
        'avg_n_sel': np.mean(fold_n_sel),
        'y_true': np.array(all_y_true),
        'y_score': np.array(all_y_score),
        'fold_probabilities': fold_probabilities,
        'gen_aucs_per_fold': gen_aucs_per_fold,
    }

def run_ga_experiment_on_data(file_path, method, classifier_type, random_states, out_dir=None):
    """執行完整 5-Fold 分層交叉驗證，使用平行化派發多個隨機種子。"""
    X, y, cases = load_data(file_path)
    
    # 為每個 random_state 生成獨立的 RNG 子種子
    child_seeds = spawn_rng_seeds(42, len(random_states))
    
    print(f"  🚀 正在派發 {len(random_states)} 個隨機種子至 {N_CPU_WORKERS} 個 CPU 工作程序...")
    
    # 🧠 平行化派發：每個 random_state 由一個獨立 Worker 處理
    rs_results = Parallel(n_jobs=N_CPU_WORKERS, verbose=10)(
        delayed(_worker_single_rs)(
            method, X, y, rs, child_seed, classifier_type, out_dir
        )
        for rs, child_seed in zip(random_states, child_seeds)
    )
    
    # 🎨 主程序中繪製每個 RS 的進化曲線（matplotlib 不可在子程序中運行）
    if out_dir:
        for rs_res in rs_results:
            gen_aucs = np.array(rs_res['gen_aucs_per_fold'])
            plot_auc_generations(
                gen_aucs,
                f"{method}_RS{rs_res['random_state']}",
                classifier_type,
                out_dir=out_dir
            )
            
    # 從結果中移除不可序列化的 gen_aucs_per_fold（僅用於繪圖）
    for rs_res in rs_results:
        rs_res.pop('gen_aucs_per_fold', None)
        
    return rs_results

## 🔄 10-BinWidth 檔案全自動遍歷與基線學術檢定模組 (Kaggle 主控執行單元)

本單元實現了對 `INPUT_DIR` 下所有組寬數據的**遍歷運行、自動跳過中斷點、結果寫入 `/kaggle/working/`、DeLong ROC 基線顯著性檢定以及全套日誌儲存**！

### 💡 快速測試模式 (FAST_TEST_MODE)
- 預設啟用 `FAST_TEST_MODE = True`。這將自動重寫超參數，以極限規格跑一次（1 個 Seed, 2 個方法, 2個世代, 族群5），**可在 15 秒內完成驗證！**
- 當您確認程式碼在您的 Kaggle 或本地環境完全能無礙編譯與走通後，請將其改為 `FAST_TEST_MODE = False` 啟動完整的學術級全量訓練。

In [ ]:
# ─── 測試控制閘 ───
FAST_TEST_MODE = False  # 設為 False 啟動完整 16-Seeds、30-Generations 全量學術模式

if FAST_TEST_MODE:
    print("🔥 [警告] 當前處於快速測試模式！超參數將被限制以進行超高速煙霧測試。")
    POP_SIZE = 5
    N_GENERATIONS = 2
    PATIENCE = 2
    TEST_RANDOM_STATES = [42]
    METHODS_TO_RUN = ['MI_top_K', 'AUC_local_search']
else:
    print("🎓 [運行] 當前處於全量學術生產模式。將執行完整 GA 優化程序。")
    POP_SIZE = 30
    N_GENERATIONS = 30
    PATIENCE = 10
    TEST_RANDOM_STATES = [42, 100, 2026, 888, 777, 1234, 5678, 9999, 314, 271, 1618, 500, 2025, 404, 1337, 6789]
    METHODS_TO_RUN = ['AUC_unrestricted'] # 'MI_top_K','AUC_local_search', 'AUC_community', 'AUC_enforce'
    

# ─── 自動遍歷執行 ───
pattern = os.path.join(INPUT_DIR, "Cine_output_*_binWidth_*.csv")
files = sorted(glob.glob(pattern))

if not files:
    print(f"❌ [錯誤] 在 {INPUT_DIR} 找不到符合檔名的重採樣特徵 CSV 檔案！")
else:
    # 1. 解析與排序所有組寬
    binwidth_files = []
    for f in files:
        match = re.search(r'binWidth_(\d+)', os.path.basename(f))
        if match:
            binwidth_files.append((int(match.group(1)), f))
    binwidth_files.sort(key=lambda x: x[0])
    
    # 🎯 【最小修改位置】只需加入下面這一行，過濾出 binWidth 介於 40 到 50 之間的配置
    # binwidth_files = [item for item in binwidth_files if 45 <= item[0] <= 50]
    
    total_configs = len(binwidth_files)
    print(f"🧭 發現共計 {total_configs} 個組寬配置檔案待處理。")
    
    # 2. 開始組寬循環
    for idx, (binwidth, file_path) in enumerate(binwidth_files, 1):
        out_dir = os.path.join(OUTPUT_DIR, f"ga_binWidth_{binwidth}")
        results_log_file = os.path.join(out_dir, 'results_log.csv')
        
        # 中斷點續傳判斷
        if os.path.isdir(out_dir) and os.path.exists(results_log_file):
            print(f"\n⏭️  [{idx}/{total_configs}] binWidth {binwidth} 的實驗記錄已存在。自動跳過。")
            continue
            
        print(f"\n▶️  [{idx}/{total_configs}] 正在處理 binWidth: {binwidth} ...")
        os.makedirs(out_dir, exist_ok=True)
        
        # 3. 運行所有指定 GA 方法
        all_results = {}
        for method in METHODS_TO_RUN:
            print(f"\n--- [方法優化] 正在執行: {method} ---")
            rs_results = run_ga_experiment_on_data(
                file_path=file_path,
                method=method,
                classifier_type=CLASSIFIER_TYPE,
                random_states=TEST_RANDOM_STATES,
                out_dir=out_dir
            )
            
            # 記錄與寫入 CSV / JSON 日誌
            save_results_csv(rs_results, out_dir)
            save_results_detail(rs_results, out_dir)
            
            all_results[method] = {
                'mean_auc': np.mean([r['mean_auc'] for r in rs_results]),
                'per_rs_results': rs_results
            }
            
        # 4. 學術顯著性檢定：DeLong ROC 檢定 (將各個 GA 方法與傳統 MI 進行對比)
        if 'MI_top_K' in all_results:
            baseline = all_results['MI_top_K']['per_rs_results']
            ga_methods = [m for m in METHODS_TO_RUN if m != 'MI_top_K']
            
            print(f"\n=== DeLong 顯著性檢定 (vs MI 基線) ===")
            delong_rows = []
            for method in ga_methods:
                ga_res = all_results[method]['per_rs_results']
                z_list, p_list = [], []
                for bl, ga in zip(baseline, ga_res):
                    if len(bl['y_true']) == len(ga['y_true']):
                        z, p, auc1, auc2 = delong_roc_test(bl['y_true'], bl['y_score'], ga['y_score'])
                        z_list.append(z)
                        p_list.append(p)
                        delong_rows.append({
                            'method': method,
                            'random_state': bl['random_state'],
                            'z': z, 'p': p,
                            'auc_mi': auc1, 'auc_ga': auc2,
                            'sig': '*' if p < 0.05 else '',
                        })
                if z_list:
                    mean_z = np.mean(z_list)
                    mean_p = np.mean(p_list)
                    sig = "p<0.05" if mean_p < 0.05 else "n.s."
                    print(f"  {method:<20} vs MI 基線: mean_z={mean_z:+.4f}, mean_p={mean_p:.4f} ({sig})")
            
            if delong_rows:
                pd.DataFrame(delong_rows).to_csv(os.path.join(out_dir, 'delong_results.csv'), index=False)
                
        # 5. 序列化全量 Pickled 結果存檔
        pkl_path = os.path.join(out_dir, 'all_results.pkl')
        with open(pkl_path, 'wb') as f:
            pickle.dump(all_results, f)
            
        print(f"\n🎉 binWidth {binwidth} 實驗運行完畢！全套產出與日誌已完全寫入至: {out_dir}/")
    print("\n🏁 恭喜！所有組寬配置特徵檔案進化流水線全部執行完畢！")